# Surgical CVS AI — Colab end-to-end pipeline

Run the cells **top to bottom** in a single session. This trains **every model
in `SEG_MODELS` (U-Net + SAM2 by default)**, then the CVS classifier, then the
benchmark — with **no Google Drive** (datasets, model cache and checkpoints all
live on the runtime's local disk `/content`).

> **First time? Run `06_smoke_test.ipynb` instead** — it runs this same
> pipeline at `limit_batches=20` in a few minutes to confirm everything is
> wired before you commit to the full multi-hour run.

Every cell is **idempotent**: if a cell stops but the runtime is still alive,
re-run from the top — the repo updates in place, the CholecSeg8k download is
cached, and training **resumes from each model's `last.ckpt`**. A *full*
runtime reset wipes `/content`, so you start over.

**Prerequisites**
- GPU runtime: Runtime -> Change runtime type -> GPU (T4 or better).
- ~12 GB free disk (CholecSeg8k ~3.1 GB + Endoscapes ~5.9 GB).
- Time: the full run trains SAM2 @ 1024 — **hours on a T4**. Use a smaller
  `epochs=` or the smoke notebook first if you just want to see it work.
- Set the branch in the clone cell if the feature branch isn't merged to `main`.

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
print("working dir:", os.getcwd())

In [ ]:
# Installs dependencies and removes the stale preinstalled torchao.
!bash scripts/colab_setup.sh

## 1. Data

Both datasets download to the local disk (`/content`) — no Drive.

- **CholecSeg8k** (~3.1 GB) -> HuggingFace Hub into `./data/cholecseg8k` (cached;
  re-running this cell is cheap once cached).
- **Endoscapes2023** (~5.9 GB) -> `/content/endoscapes2023` from the CAMMA public
  mirror (`wget -c` resumes a partial download).

In [ ]:
# CholecSeg8k (~3.1 GB) -> local HuggingFace cache (cheap once cached)
!bash scripts/download_cholecseg8k.sh

# Endoscapes2023 (~5.9 GB) -> /content (downloaded each runtime; not on Drive)
!wget -q -c -O /content/endoscapes.zip https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
!unzip -q -o /content/endoscapes.zip -d /content/endoscapes2023

## 2. Verify the data layer

Builds the video-level splits for both datasets. Non-zero frame counts
(CholecSeg8k summing to 8080) mean the loaders work.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset
from src.data.endoscapes import Endoscapes2023Dataset

for split in ("train", "val", "test"):
    print(f"CholecSeg8k {split:5s}: {len(CholecSeg8kDataset(split=split))} frames")
for split in ("train", "val", "test"):
    ds = Endoscapes2023Dataset(root="/content/endoscapes2023", split=split)
    print(f"Endoscapes  {split:5s}: {len(ds)} CVS keyframes")

## 3. Choose which models to run

`SEG_MODELS` is the list of segmentation models the pipeline **trains and
benchmarks**. Edit it to add more — e.g. a published SOTA model — and the
benchmark table gains one row per model.

**Adding a new segmentation model (3 steps):**
1. `configs/model/<name>.yaml` — its hyperparameters (copy `unet_baseline.yaml`
   as a template, set `name: <name>`).
2. `src/train/train_segmentation.py` -> `build_model()` — add an
   `elif name == "<name>":` branch returning the model. It must output
   `(B, num_classes, H, W)` logits and expose
   `param_groups(lr_encoder, lr_decoder)` (see `UNetBaseline` for the contract).
3. `configs/benchmark.yaml` -> add a `models:` entry, and append `"<name>"` to
   `SEG_MODELS` below.

`CVS_SEG_MODEL` picks which *trained* segmentation model supplies the frozen
6-channel masks for the CVS stage (the primary SAM2 by default).

In [ ]:
# Segmentation models to train + benchmark (add more here — see the cell above).
SEG_MODELS = ["unet_baseline", "sam2_lora"]

# Trained segmentation model that feeds the CVS classifier (frozen mask source).
CVS_SEG_MODEL = "sam2_lora"

assert CVS_SEG_MODEL in SEG_MODELS, "CVS_SEG_MODEL must be one of SEG_MODELS"
print("will train:", SEG_MODELS, "| CVS uses:", CVS_SEG_MODEL)

## 4. Train the segmentation models

Trains each model in `SEG_MODELS` with the full config (default `epochs=100`,
`low_memory=true` for 16 GB T4s, class-balanced loss + weighted sampler). Each
model's best checkpoint is written to `outputs/<model>/best.ckpt`.

Resumable per model — re-running continues each from its `last.ckpt`.

- Faster trial: append `epochs=5` (or use the smoke notebook).
- The pre-training class-frequency pass walks the whole train split once
  (slow, no progress bar) before the Lightning bar appears — that's expected.

In [ ]:
for model in SEG_MODELS:
    print(f"\n{'='*60}\n  training segmentation: {model}\n{'='*60}")
    !python -u -m src.train.train_segmentation model={model}

## 5. Train the CVS classifier

Freezes `CVS_SEG_MODEL`'s checkpoint (from step 4) as the 6-channel mask
source, then trains the 3-criteria CVS classifier on Endoscapes2023. The best
checkpoint is written to `outputs/cvs_classifier/best.ckpt`. Also resumable.

In [ ]:
cmd = (
    "python -u -m src.train.train_cvs "
    "data.root=/content/endoscapes2023 "
    f"segmentation.model_config=configs/model/{CVS_SEG_MODEL}.yaml "
    f"segmentation.checkpoint=outputs/{CVS_SEG_MODEL}/best.ckpt"
)
print(cmd)
!{cmd}

## 6. Benchmark

`benchmark_runner` evaluates every trained segmentation checkpoint on the
CholecSeg8k test split and writes the comparison table (mIoU, Cystic Duct Dice
with 95% bootstrap CIs) to `results/benchmark_table.md`. Models still missing a
checkpoint show as `TBD`.

In [ ]:
!python -u -m src.eval.benchmark_runner

## 7. Results

Numeric comparison table + a qualitative look at each model's predictions on a
few validation frames. (Scores reflect however long you trained — short runs
look rough; that's fine.) CVS test metrics (`test_map`, `test_qwk`) are printed
at the end of step 5's output.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf
from IPython.display import Markdown, display

from src.train.train_segmentation import build_model
from src.eval.benchmark_runner import load_segmentation_checkpoint
from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES
from src.data.transforms import build_eval_transforms

# --- numeric: the benchmark table ---
table = Path("results/benchmark_table.md")
if table.is_file():
    display(Markdown(table.read_text()))

# --- qualitative: predictions per trained segmentation model ---
IMG = 512
device = "cuda" if torch.cuda.is_available() else "cpu"
ds = CholecSeg8kDataset(split="val", image_size=IMG,
                        transform=build_eval_transforms(IMG))
samples = [ds[i] for i in range(3)]

for model_name in SEG_MODELS:
    ckpt = f"outputs/{model_name}/best.ckpt"
    if not Path(ckpt).is_file():
        print(f"[skip] {model_name}: no checkpoint at {ckpt}")
        continue
    cfg = OmegaConf.load(f"configs/model/{model_name}.yaml")
    model = load_segmentation_checkpoint(build_model(cfg, pretrained=False), ckpt)
    model = model.to(device).eval()

    fig, ax = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
    fig.suptitle(model_name, fontsize=14)
    for i, s in enumerate(samples):
        with torch.no_grad():
            pred = model(s["image"].unsqueeze(0).to(device)).argmax(dim=1)[0].cpu()
        rgb = s["image"].permute(1, 2, 0).numpy()
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
        ax[i, 0].imshow(rgb)
        ax[i, 1].imshow(s["mask"], vmin=0, vmax=5, cmap="tab10")
        ax[i, 2].imshow(pred, vmin=0, vmax=5, cmap="tab10")
        for a in ax[i]:
            a.axis("off")
    ax[0, 0].set_title("input")
    ax[0, 1].set_title("ground truth")
    ax[0, 2].set_title(f"{model_name} prediction")
    plt.tight_layout()
    plt.show()

print("6 classes:", CLASS_NAMES)
# Interactive demo (uncomment to launch a blocking Gradio server):
# !python -m app.gradio_demo